In [1]:
import torch
import train

# Checking models performance on small number of epochs

## Round 1

In [2]:
import os, json
import plotly.graph_objects as go
def open_selected(prefix, suffix, index_list=None, path=train.save_path):
    l = []
    path = os.path.join(path, 'tmp')

    if index_list is None:
        i = 0
        path = os.path.join(os.path.dirname(path), prefix + str(i) + suffix)
        while os.path.exists(path):
            with open(path, 'r') as f:
                data = json.load(f)
            l.append(data)

            i += 1
            path = os.path.join(os.path.dirname(path), prefix + str(i) + suffix)
    else:
        for i in index_list:
            path = os.path.join(os.path.dirname(path), prefix + str(i) + suffix)
            with open(path, 'r') as f:
                data = json.load(f)
            l.append(data)
    return l

In [3]:
def visualise_losses(index_list=None, path=train.save_path):
    losses_list = open_selected(prefix="loss", suffix=".json", index_list=index_list, path=path)
    if index_list is None:
        index_list = list(range(len(losses_list)))
    
    fig = go.Figure()
    for i, loss in zip(index_list, losses_list):
        x = list(range(len(loss)))
        fig.add_trace(go.Scatter(x=x, y=loss, mode="lines", name=f"Model {i}"))
    
    fig.update_layout(
        title="Cross Entropy Loss Of Models",
        xaxis_title="Batch",
        yaxis_title="Cross Entropy Loss",
        template="plotly_dark"
    )
    fig.show()

In [4]:
from plotly.subplots import make_subplots
from sklearn.metrics import f1_score, accuracy_score
import numpy as np
def visualise_metrics(index_list=None, path=train.save_path):
    vals = open_selected(prefix="validation", suffix=".json", index_list=index_list, path=path)
    if index_list is None:
        index_list = list(range(len(vals)))
    labels = [f"Model {i}" for i in index_list]

    fig = make_subplots(rows=2, cols=3, subplot_titles=["Accuracy", "F1-Score Micro", "F1-Score Macro", "F1-Score Min", "F1-Score Max"])

    accuracies = [accuracy_score(i["expected"], i["predicted"]) for i in vals]
    f1_micros = [f1_score(i["expected"], i["predicted"], average="micro") for i in vals]
    f1_macros = [f1_score(i["expected"], i["predicted"], average="macro") for i in vals]
    f1_mins = []
    f1_maxs = []
    for val in vals:
        predicted = val['predicted']
        expected = val['expected']

        f1 = f1_score(predicted, expected, average=None)
        f1_min = min(f1)
        f1_max = max(f1)

        f1_mins.append(f1_min)
        f1_maxs.append(f1_max)

    fig.add_trace(go.Bar(
        x=labels,
        y=accuracies, name="Accuracy"
    ), row=1, col=1)
    fig.add_trace(go.Bar(
        x=labels,
        y=np.ones(len(accuracies)) * 1/26, name="Accuracy (Baseline)"
    ), row=1, col=1)
    fig.add_trace(go.Bar(
        x=labels,
        y=f1_micros, name="F1 Micro"
    ), row=1, col=2)
    fig.add_trace(go.Bar(
        x=labels,
        y=np.ones(len(accuracies)) * 1/26, name="F1 Micro (Baseline)"
    ), row=1, col=2)
    fig.add_trace(go.Bar(
        x=labels,
        y=f1_macros, name="F1 Macro"
    ), row=1, col=3)
    fig.add_trace(go.Bar(
        x=labels,
        y=np.ones(len(accuracies)) * 1/26, name="F1 Macro (Baseline)"
    ), row=1, col=3)

    fig.add_trace(go.Bar(
        x=labels,
        y=f1_mins, name="Min F1-Score Across Labels"
    ), row=2, col=1)
    fig.add_trace(go.Bar(
        x=labels,
        y=np.ones(len(accuracies)) * 1/26, name="Min F1-Score Across Labels (Baseline)"
    ), row=2, col=1)
    fig.add_trace(go.Bar(
        x=labels,
        y=f1_maxs, name="Max F1-Score Across Labels"
    ), row=2, col=2)
    fig.add_trace(go.Bar(
        x=labels,
        y=np.ones(len(accuracies)) * 1/26, name="Max F1-Score Across Labels (Baseline)"
    ), row=2, col=2)
    
    y_range = [0.0, 1.0]
    fig.update_xaxes(title_text="Models", row=1, col=1)
    fig.update_yaxes(title_text="Accuracy", row=1, col=1, range=y_range)
    fig.update_xaxes(title_text="Models", row=1, col=2)
    fig.update_yaxes(title_text="F1-Micro", row=1, col=2, range=y_range)
    fig.update_xaxes(title_text="Models", row=1, col=3)
    fig.update_yaxes(title_text="F1-Macro", row=1, col=3, range=y_range)

    fig.update_xaxes(title_text="Models", row=2, col=1)
    fig.update_yaxes(title_text="F1-Min", row=2, col=1, range=y_range)
    fig.update_xaxes(title_text="Models", row=2, col=2)
    fig.update_yaxes(title_text="F1-Max", row=2, col=2, range=y_range)

    fig.update_layout(
        barmode='overlay',
        title="Metrics Comparison Of Models",
        template="plotly_dark",
        height=1000
    )
    fig.show()

In [5]:
from sklearn.metrics import confusion_matrix
def visualise_label_accuracies(index_list=None, path=train.save_path):
    vals = open_selected(prefix="validation", suffix=".json", index_list=index_list, path=path)
    if index_list is None:
        index_list = list(range(len(vals)))
    models_no = [f"Model {i}" for i in index_list]
    labels = [chr(ord('a') + i) for i in range(26)]

    cms = [confusion_matrix(val['expected'], val['predicted']) for val in vals]
    label_accs = [cm.diagonal() / cm.sum(axis=1) for cm in cms]
    
    fig = go.Figure()
    fig.add_trace(
        go.Bar(
            x = labels,
            y = np.ones(26, dtype=np.float64) * 1/26,
            name=f"Random classifier (baseline)"
            )
    )

    for i, acc in enumerate(label_accs):
        fig.add_trace(go.Bar(
            x=labels,
            y=acc,
            name=f"Model {models_no[i]}",
            visible=(i==0)
        ))
    buttons = [
        {
            "label": f"Matrix {i}",
            "method": "update",
            "args": [{"visible": [True] + [j == i for j in range(len(label_accs))]}]
        }
        for i in range(len(label_accs))
    ]
    
    fig.update_layout(
        barmode='stack',
        title="Accuracies Per Label",
        xaxis_title="Label",
        yaxis_title="Accuracy",
        updatemenus=[{
        'buttons': buttons,
        'direction': 'down',
        'showactive': True,
        'xanchor': 'center',
        'yanchor': 'top'
        }],
        template="plotly_dark"
    )
    fig.update_yaxes(range=[0.0, 1.0])
    fig.show()

In [6]:
n = 10
c_low = 5
c_high = 15
num_l = 2
exp_low = -13
exp_high = -4
epochs = 2

In [7]:
seed = 123456789
batch_size = 128

In [7]:
models, alphas = train.models_rand(n, c_low, c_high, exp_low, exp_high)
hyperparams_list = [{"lr": a} for a in alphas]

for i, (model, hyperparams) in enumerate(zip(models, hyperparams_list)):
    print(f"Training model {i}; channels: {model.summary}; hyperparams: {hyperparams}")
    train.train(model, batch_size=batch_size, hyperparams=hyperparams, epochs=epochs, seed=seed)

Training model 0; channels: [8, 12]; hyperparams: {'lr': 0.00048828125}


c:\Users\mokrota\Documents\GitHub\Neural-Network-Project\.venv\Lib\site-packages\torch\nn\modules\conv.py:549: UserWarning: Using padding='same' with even kernel lengths and odd dilation may require a zero-padded copy of the input be created (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\native\Convolution.cpp:1037.)
  return F.conv2d(
c:\Users\mokrota\Documents\GitHub\Neural-Network-Project\.venv\Lib\site-packages\torch\nn\modules\module.py:1739: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  return self._call_impl(*args, **kwargs)


Epoch 1, batch 100, loss 3.221
Epoch 1, batch 200, loss 3.058
Epoch 1, batch 300, loss 2.965
Epoch 1, batch 400, loss 2.904
Epoch 1, batch 500, loss 2.864
Epoch 1, batch 600, loss 2.849
Epoch 1, batch 700, loss 2.820
Epoch 2, batch 100, loss 2.787
Epoch 2, batch 200, loss 2.771
Epoch 2, batch 300, loss 2.749
Epoch 2, batch 400, loss 2.731
Epoch 2, batch 500, loss 2.715
Epoch 2, batch 600, loss 2.701
Epoch 2, batch 700, loss 2.689
Training model 1; channels: [7, 12]; hyperparams: {'lr': 0.0009765625}
Epoch 1, batch 100, loss 3.196
Epoch 1, batch 200, loss 3.049
Epoch 1, batch 300, loss 2.978
Epoch 1, batch 400, loss 2.921
Epoch 1, batch 500, loss 2.867
Epoch 1, batch 600, loss 2.849
Epoch 1, batch 700, loss 2.818
Epoch 2, batch 100, loss 2.802
Epoch 2, batch 200, loss 2.790
Epoch 2, batch 300, loss 2.769
Epoch 2, batch 400, loss 2.749
Epoch 2, batch 500, loss 2.745
Epoch 2, batch 600, loss 2.739
Epoch 2, batch 700, loss 2.732
Training model 2; channels: [7, 10]; hyperparams: {'lr': 0.00

In [ ]:
visualise_losses()

This is much better, although it is still noisy increasing batch size would hinder speed of computation so we've decided to stick with this

Now let's check performance via confusion matrix

In [9]:
# load if necessary
def load_model(folder, model_i):
    metadata_path = os.path.dirname(train.save_path)
    model_path = os.path.join(metadata_path, folder, f'cnn{model_i}.pth')
    metadata_path = os.path.join(metadata_path, folder, f"train_metadata{model_i}.json")
    with open(metadata_path, 'r') as f:
        metadata = json.load(f)
    
    model = train.ConvNetPooling(train.height, train.width, train.output_size, channels=metadata['channels'])
    model.load_state_dict(torch.load(model_path, weights_only=True))
    return model

# folder = "training_pyramid3"
# models = [load_model(folder, i) for i in range(5, 10)]

In [8]:
visualise_label_accuracies(list(range(10)), path=os.path.abspath("./model/training_reverse_pyramid1"))

In [9]:
visualise_metrics(list(range(10)), path=os.path.abspath("./model/training_reverse_pyramid1"))

# Training

Now that we've decided to stick with model 3 let's train it for more epochs

In [12]:
no = 0
hyperparams = {'lr': 0.00048828125}
model = load_model("training_reverse_pyramid1", str(no))

In [13]:
seed = 123456789
batch_size = 128

In [14]:
epochs = 20
train.train(model=model, hyperparams=hyperparams, batch_size=batch_size, epochs=epochs)

c:\Users\mokrota\Documents\GitHub\Neural-Network-Project\.venv\Lib\site-packages\torch\nn\modules\module.py:1739: UserWarning:

Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.



Epoch 1, batch 100, loss 2.678
Epoch 1, batch 200, loss 2.674
Epoch 1, batch 300, loss 2.665
Epoch 1, batch 400, loss 2.655
Epoch 1, batch 500, loss 2.648
Epoch 1, batch 600, loss 2.644
Epoch 1, batch 700, loss 2.637
Epoch 2, batch 100, loss 2.634
Epoch 2, batch 200, loss 2.632
Epoch 2, batch 300, loss 2.631
Epoch 2, batch 400, loss 2.630
Epoch 2, batch 500, loss 2.617
Epoch 2, batch 600, loss 2.617
Epoch 2, batch 700, loss 2.617
Epoch 3, batch 100, loss 2.610
Epoch 3, batch 200, loss 2.612
Epoch 3, batch 300, loss 2.605
Epoch 3, batch 400, loss 2.605
Epoch 3, batch 500, loss 2.604
Epoch 3, batch 600, loss 2.601
Epoch 3, batch 700, loss 2.601
Epoch 4, batch 100, loss 2.599
Epoch 4, batch 200, loss 2.593
Epoch 4, batch 300, loss 2.589
Epoch 4, batch 400, loss 2.592
Epoch 4, batch 500, loss 2.588
Epoch 4, batch 600, loss 2.585
Epoch 4, batch 700, loss 2.589
Epoch 5, batch 100, loss 2.584
Epoch 5, batch 200, loss 2.581
Epoch 5, batch 300, loss 2.583
Epoch 5, batch 400, loss 2.583
Epoch 5,

In [ ]:
visualise_losses([10])

In [10]:
visualise_label_accuracies([10], path=os.path.abspath("./model/training_reverse_pyramid1"))
visualise_metrics([10], path=os.path.abspath("./model/training_reverse_pyramid1"))